In [28]:
import os
import sys
import docx2txt
import nltk
import spacy
import re
import requests
import subprocess
import pandas as pd
from tabulate import tabulate
from pdfminer.high_level import extract_text
from PyPDF2 import PdfReader, PdfFileWriter, PdfFileMerger

nlp = spacy.load('en_core_web_sm')


#Extracting text from DOCX
def extract_text_from_docx(docx_path):
    txt = docx2txt.process(docx_path)
    if txt:
        return txt.replace('\t', ' ')
    return None

## Extracting text from PDF
def extract_text_from_pdf(pdf_path):
    return extract_text(pdf_path)


def extract_names(txt):
    person_names = []
    for sent in nltk.sent_tokenize(txt):
        for chunk in nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize(sent))):
            if hasattr(chunk, 'label') and chunk.label() == 'PERSON':
                person_names.append(
                    ' '.join(chunk_leave[0] for chunk_leave in chunk.leaves())
                )
    return person_names[0]


PHONE_REG = re.compile(
    r'(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?)?\d{3,4}[-.\s]?\d{4}'
)

def extract_phone_number(text):
    match = PHONE_REG.search(text)
    if match:
        return match.group().strip()
    return None

EMAIL_REG = re.compile(r'[a-z0-9\.\-+_]+@[a-z0-9\.\-+_]+\.[a-z]+')
 
def extract_emails(resume_text):
    return re.findall(EMAIL_REG, resume_text)

SKILLS_DB = [
    'andriod developer',
    'app developer',
    'Javascript',
    'Java',
    'machine learning',
    'data science',
    'python',
    'CSS',
    'doctor',
    'teacher',
    'web development',
    'communication',
    'team work',
    'sql',
    'html'
]

def extract_skills(input_text):
    stop_words = set(nltk.corpus.stopwords.words('english'))
    word_tokens = nltk.tokenize.word_tokenize(input_text)
    
    # remove the stop words
    filtered_tokens = [w for w in word_tokens if w not in stop_words]
 
    # remove the punctuation
    filtered_tokens = [w for w in word_tokens if w.isalpha()]
    
    # generate bigrams and trigrams (such as artificial intelligence)
    bigrams_trigrams = list(map(' '.join, nltk.everygrams(filtered_tokens, 2, 3)))
 
    # we create a set to keep the results in.
    found_skills = set()

    # we search for each token in our skills database
    for token in filtered_tokens:
        if token.lower() in SKILLS_DB:
            found_skills.add(token)
 
    # we search for each bigram and trigram in our skills database
    for ngram in bigrams_trigrams:
        if ngram.lower() in SKILLS_DB:
            found_skills.add(ngram)
    return found_skills
#========================Extracting education=========================================

RESERVED_WORDS = [
    'school',
    'college',
    'university',
    'academy',
    'faculty',
    'degree',
    'institute'
]

def extract_education(text):
    education = set()

    edu_block = re.split(
        r'\n(?:experience|skills|projects|certifications|activities|awards|honors)\b',
        text,
        flags=re.IGNORECASE
    )[0]

    for line in edu_block.split('\n'):
        line = line.strip()

        if not line or len(line) < 10:
            continue
        if line.startswith(('•', '-', '*')):
            continue

        lower = line.lower()

        if not any(word in lower for word in RESERVED_WORDS):
            continue

        if any(x in lower for x in [
            'scholarship', 'grantee', 'award', 'honor', 'honours', 'dean'
        ]):
            continue

        if any(x in lower for x in [
            'bachelor', 'master', 'senior high', 'high school education',
            'strand', 'track', 'science', 'technology', 'engineering'
        ]):
            continue

        line = re.sub(r'[.\u2026\u22EF\u00B7\u2219\u2022]{2,}.*', '', line)
        line = re.sub(r',.*', '', line)
        line = re.sub(r'\(.*?\)', '', line)

        line = re.sub(r'\s{2,}', ' ', line).strip()

        if 20 <= len(line) <= 70:
            education.add(line)

    return education

DEGREE_KEYWORDS = [
    'bachelor', 'bachelors', 'b.s.', 'b.s', 'bs', 'b.a.', 'b.a', 'ba',
    'master', 'masters', 'm.s.', 'm.s', 'ms', 'm.a.', 'm.a', 'ma', 'mba',
    'phd', 'ph.d', 'doctorate',
    'associate', 'a.s.', 'a.a.',
    'btech', 'b.tech', 'mtech', 'm.tech',
    'bsc', 'b.sc', 'msc', 'm.sc',
    'beng', 'b.eng', 'meng', 'm.eng',
    'diploma'
]

DEGREE_PATTERNS = [
    r'(?:bachelor|bachelors?)(?:\s+of)?\s+(?:science|arts|engineering|technology|business|commerce|education|fine\s+arts)(?:\s+in\s+[\w\s]+)?',
    r'(?:master|masters?)(?:\s+of)?\s+(?:science|arts|engineering|technology|business|administration|commerce|education)(?:\s+in\s+[\w\s]+)?',
    r'b\.?s\.?(?:\s+(?:in|of))?\s+[\w\s]+',
    r'm\.?s\.?(?:\s+(?:in|of))?\s+[\w\s]+',
    r'b\.?a\.?(?:\s+(?:in|of))?\s+[\w\s]+',
    r'm\.?a\.?(?:\s+(?:in|of))?\s+[\w\s]+',
    r'mba(?:\s+in\s+[\w\s]+)?',
    r'ph\.?d\.?(?:\s+(?:in|of))?\s+[\w\s]+',
    r'associate\s+(?:degree\s+)?(?:of|in)?\s*(?:science|arts)?(?:\s+in\s+[\w\s]+)?',
    r'diploma\s+in\s+[\w\s]+'
]

def extract_degree(text):
    degrees = set()
    
    text_lower = text.lower()
    
    edu_section = text
    edu_match = re.search(
        r'(education|academic|qualification|degrees?)[\s\S]*?(?=\n(?:experience|skills|projects|certifications|work|employment|activities)|$)',
        text_lower,
        re.IGNORECASE
    )
    
    if edu_match:
        edu_section = text[edu_match.start():edu_match.end()]
    
    for pattern in DEGREE_PATTERNS:
        matches = re.finditer(pattern, edu_section, re.IGNORECASE)
        for match in matches:
            degree_text = match.group().strip()
            degree_text = re.sub(r'\s+', ' ', degree_text)
            degree_text = re.sub(r'[•\-*\(\)]', '', degree_text).strip()
            
            degree_text = re.sub(r'^[\s,;:\-]+|[\s,;:\-]+$', '', degree_text)
            
            if 3 <= len(degree_text) <= 100:
                degrees.add(degree_text.title())
    
    if not degrees:
        lines = edu_section.split('\n')
        for line in lines:
            line_lower = line.lower().strip()
            
            if not line_lower or len(line_lower) < 5:
                continue
            
            if any(keyword in line_lower for keyword in DEGREE_KEYWORDS):
                cleaned_line = re.sub(r'[•\-*]', '', line).strip()
                cleaned_line = re.sub(r'\s+', ' ', cleaned_line)
                cleaned_line = re.sub(r'\d{4}\s*-?\s*\d{0,4}', '', cleaned_line).strip()  # Remove years
                
                if 5 <= len(cleaned_line) <= 100:
                    degrees.add(cleaned_line)
    
    return list(degrees) if degrees else ['Not specified']



def extract_experience(text):
    """
    Extract both work experience and projects with concise descriptions
    """
    entries = []
    
    sections_pattern = r'((?:professional\s+)?(?:work\s+)?experience|projects?|personal\s+projects?)[\s\S]*?(?=\n(?:education|skills|certifications|memberships|organizations|certificates|awards|scholarships|references)|$)'
    
    matches = re.finditer(sections_pattern, text, re.IGNORECASE)
    
    for match in matches:
        section_text = match.group()
        
        # Remove unwanted subsections
        section_text = re.split(
            r'\n(?:memberships|organizations|significant\s+certificates|certifications|awards|scholarships)',
            section_text,
            flags=re.IGNORECASE
        )[0]
        
        lines = section_text.split('\n')
        current_entry = {'title': '', 'details': []}
        
        for line in lines:
            line = line.strip()
            
            if not line:
                if current_entry['title']:
                    # Combine title with top 2-3 key details only
                    concise_entry = current_entry['title']
                    if current_entry['details']:
                        # Take only first 2 most important details
                        key_details = current_entry['details'][:2]
                        concise_entry += ' - ' + ' '.join(key_details)
                    
                    # Limit length to ~200 characters
                    if len(concise_entry) > 200:
                        concise_entry = concise_entry[:197] + '...'
                    
                    if len(concise_entry) > 20:
                        entries.append(concise_entry)
                    
                    current_entry = {'title': '', 'details': []}
                continue
            
            if re.match(r'^\s*(?:professional\s+)?(?:work\s+)?(?:experience|projects?|personal\s+projects?)$', line, re.IGNORECASE):
                continue
            
            cleaned_line = re.sub(r'^[•\-*●○►]\s*', '', line)
            
            is_title = (
                not current_entry['title'] or  # First line
                (line.startswith(('•', '-', '*')) and len(line) > 30) or  # Bullet with substantial text
                (line[0].isupper() and any(kw in line.lower() for kw in 
                    ['app', 'system', 'model', 'analysis', 'prediction', 'project', 
                     'developer', 'analyst', 'intern', 'using', 'through']))
            )
            
            if is_title:
                # Save previous entry if exists
                if current_entry['title']:
                    concise_entry = current_entry['title']
                    if current_entry['details']:
                        key_details = current_entry['details'][:2]
                        concise_entry += ' - ' + ' '.join(key_details)
                    
                    if len(concise_entry) > 200:
                        concise_entry = concise_entry[:197] + '...'
                    
                    if len(concise_entry) > 20:
                        entries.append(concise_entry)
                
                current_entry = {'title': cleaned_line, 'details': []}
            else:
                if len(cleaned_line) > 15 and len(current_entry['details']) < 3:
                    current_entry['details'].append(cleaned_line)
        
        if current_entry['title']:
            concise_entry = current_entry['title']
            if current_entry['details']:
                key_details = current_entry['details'][:2]
                concise_entry += ' - ' + ' '.join(key_details)
            
            if len(concise_entry) > 200:
                concise_entry = concise_entry[:197] + '...'
            
            if len(concise_entry) > 20:
                entries.append(concise_entry)
    
    seen = set()
    unique_entries = []
    for entry in entries:
        if entry not in seen:
            seen.add(entry)
            unique_entries.append(entry)
    
    return unique_entries if unique_entries else ['No experience or projects found']





    

def parse_resume(resume_text, output_file_path):
    name = extract_names(resume_text)
    email = extract_emails(resume_text)
    phone = extract_phone_number(resume_text)
    skills = extract_skills(resume_text)
    education = extract_education(resume_text)
    degree = extract_degree(resume_text)
    experience=extract_experience(resume_text)
    
    name = name or ['']
    email = email or ['']
    phone = phone or ['']
    skills = skills or ['']
    education = education or ['']
    degree = degree or ['']
    experience=experience or ['']
    info = {
        "Name": name,
        "Email": email,
        "Phone": phone,
        "Skills": skills,
        "Education":education,
        "Degree":degree,
        "Experience":experience,
    }
    with open(output_file_path, 'a', encoding='utf-8') as f:
        f.write(tabulate(info, headers="keys"))
        f.write("\n\n")
    return info

resume_list=[]
if __name__ == '__main__':
    directory = 'C:/Users/Audrie Lex Afundar/Downloads/resume_data/Resumes'
    for filename in os.listdir(directory):
        output_file_path = 'C:/Users/Audrie Lex Afundar/Downloads/resume_data/resume_parsed.txt'
        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path) and file_path.endswith('.pdf'):
            resume_text = extract_text_from_pdf(file_path)
            print('File_name:', filename)
            resume_list.append(parse_resume(resume_text,output_file_path))
        elif os.path.isfile(file_path) and file_path.endswith('.docx'):
            resume_text = extract_text_from_docx(file_path)
            print('-------------File_name:-------------', filename)
            resume_list.append(parse_resume(resume_text,output_file_path))
        else:
            print('Unsupported File Format:', filename)
        continue

-------------File_name:------------- Abiral_Pandey_Fullstack_Java.docx
-------------File_name:------------- Achyuth Resume_8.docx
-------------File_name:------------- Adelina_Erimia_PMP1.docx
-------------File_name:------------- Adhi Gopalam - SM.docx
-------------File_name:------------- AjayKumar.docx
-------------File_name:------------- Akhil.profile.docx
-------------File_name:------------- Akhil_Sr BSA.docx
-------------File_name:------------- Alekhya Resume.docx
-------------File_name:------------- Amar Sr BSA.docx
-------------File_name:------------- Ami Jape.docx
-------------File_name:------------- Amrinder Business Analyst.docx
-------------File_name:------------- Amulya Komatineni.docx
-------------File_name:------------- Anil Krishna Mogalaturthi.docx
-------------File_name:------------- AnilAgarwal.docx
-------------File_name:------------- Anudeep N_Sr Java Developer.docx
-------------File_name:------------- Ashok Jayakumar - PM.docx
-------------File_name:------------- Ash

In [8]:

job_input = {
    "skills": [], # average na skills na need
    "experience": [], # average experience na need
    "education": [], # lowest educ possible
    "job_title": [], # job title ng joborder
    "job_years":[] # average # of years na need
}


print("Enter skills (type 'done' to finish):")
while True:
    skill = input("> ")
    if skill.lower() == "done":
        break
    job_input["skills"].append(skill.strip())

print("\nEnter job experiences (type 'done' to finish):")
while True:
    exp = input("> ")
    if exp.lower() == "done":
        break
    job_input["experience"].append(exp.strip())

job_input["education"].append(input("\nEnter education: ").strip())
job_input["job_title"].append(input("Enter job title: ").strip())
job_input["job_years"].append(input("Enter job years: ").strip())

print("\nFinal job_input:")
print(job_input)

Enter skills (type 'done' to finish):


>  python
>  aws
>  R
>  excel
>  html
>  css
>  done



Enter job experiences (type 'done' to finish):


>  data analyst
>  project manager
>  web developer
>  ai learning
>  done

Enter education:  university
Enter job title:  data analyst
Enter job years:  3



Final job_input:
{'skills': ['python', 'aws', 'R', 'excel', 'html', 'css'], 'experience': ['data analyst', 'project manager', 'web developer', 'ai learning'], 'education': ['university'], 'job_title': ['data analyst'], 'job_years': ['3']}


In [36]:
def dict_to_text(d):
    parts = []

    for value in d.values():
        if isinstance(value, list) or isinstance(value, set):
            parts.append(" ".join(map(str, value)))
        elif isinstance(value, dict):
            parts.append(" ".join(map(str, value.values())))
        elif value:
            parts.append(str(value))

    return " ".join(parts).lower()




def weighted_dict_to_text(d):
    def norm(x):
        if isinstance(x, list):
            return x
        if isinstance(x, set):
            return list(x)
        if isinstance(x, str):
            return [x]
        return []

    text = []

    # ---- SKILLS (highest weight) ----
    text += norm(d.get('skills') or d.get('Skills')) * 3

    # ---- EXPERIENCE ----
    text += norm(d.get('experience') or d.get('Experience')) * 2

    # ---- JOB TITLE (job_input only, but useful) ----
    text += norm(d.get('job_title')) * 2

    # ---- EDUCATION ----
    text += norm(d.get('education') or d.get('Education'))
    text += norm(d.get('Degree'))

    # ---- YEARS (convert to text so TF-IDF sees it) ----
    years = d.get('job_years')
    if years:
        text += [str(y) for y in norm(years)]

    return " ".join(text).lower()



job_text = weighted_dict_to_text(job_input)
resume_texts = [weighted_dict_to_text(r) for r in resume_list]

type(job_text)     
type(resume_texts)


list

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer


corpus = resume_texts+ [job_text]

In [38]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

tfidf_matrix = vectorizer.fit_transform(corpus)

In [39]:
job_vector = tfidf_matrix[-1]    
resume_vectors = tfidf_matrix[:-1]

In [40]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(job_vector, resume_vectors)[0]

In [41]:
similarity_scores

array([0.09920476, 0.09707288, 0.07380156, 0.04359387, 0.0763597 ,
       0.13289255, 0.12723141, 0.1089995 , 0.12623688, 0.06757128,
       0.09976006, 0.08745304, 0.11575034, 0.06039831, 0.10507772,
       0.07854462, 0.07394551, 0.05939918, 0.06763999, 0.11810487,
       0.11076444, 0.09003105, 0.15247027, 0.10393085, 0.08780156,
       0.11781772, 0.12238728, 0.12052932, 0.11552654, 0.02316436,
       0.10750412, 0.12649498, 0.12023155, 0.11105537, 0.12541515,
       0.12260732, 0.06918684, 0.1152079 , 0.07957492, 0.0406353 ,
       0.1152079 , 0.05342798, 0.09173669, 0.01015173, 0.06133534,
       0.04077011, 0.0377531 , 0.13170963, 0.06933075, 0.08792361,
       0.13320815, 0.10325037, 0.13336481, 0.10267141, 0.12035335,
       0.12138961, 0.11303137, 0.07549586, 0.11215608, 0.09415976,
       0.18623081, 0.11141959, 0.07862807, 0.12692447, 0.22193974,
       0.10875846, 0.11542797, 0.14819773, 0.10849427, 0.16907984,
       0.11957816, 0.14110022, 0.09370527, 0.15363223, 0.12087

In [42]:
ranked = sorted(
    enumerate(similarity_scores),
    key=lambda x: x[1],
    reverse=True
)

for i, score in ranked[:5]:
    print(f"Resume {i} → Similarity: {score:.4f}")

Resume 64 → Similarity: 0.2219
Resume 211 → Similarity: 0.2120
Resume 177 → Similarity: 0.1888
Resume 60 → Similarity: 0.1862
Resume 195 → Similarity: 0.1833
